In [1]:
!pip install -q sentence-transformers faiss-cpu langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.2 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

In [3]:
from google.colab import files

uploaded = files.upload()

Saving medquad_embeddings_v3.npy to medquad_embeddings_v3.npy
Saving medquad_docs_v3.pkl to medquad_docs_v3.pkl


In [4]:
embeddings = np.load("medquad_embeddings_v3.npy")

with open("medquad_docs_v3.pkl", "rb") as f:
    small_docs = pickle.load(f)

print("Embeddings shape:", embeddings.shape)
print("Total documents:", len(small_docs))

Embeddings shape: (16407, 384)
Total documents: 16407


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 16407


In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [8]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [9]:
def classify_intent(query):
    prompt = f"""
You are a medical query intent classifier.

Classify the user's query into exactly one of these categories:

1. greeting
2. general_medical
3. symptom_question
4. treatment_question
5. medication_question
6. emergency
7. out_of_scope

Definitions:
- greeting: hello, hi, thanks, small talk
- general_medical: asking general medical information
- symptom_question: asking about symptoms or possible condition
- treatment_question: asking about treatment, prevention, management
- medication_question: asking about medicines, dosage, side effects, missed medicine
- emergency: serious urgent symptoms like chest pain, severe breathing difficulty, stroke signs, severe dehydration, confusion with diabetes, unconsciousness
- out_of_scope: not medical or not health-related

Return only the category name.

User query:
{query}

Category:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip().lower()

In [10]:
def detect_emergency(query):
    emergency_keywords = [
        "chest pain",
        "chest tightness",
        "shortness of breath",
        "difficulty breathing",
        "left arm pain",
        "jaw pain",
        "cold sweating",
        "unconscious",
        "fainting",
        "stroke",
        "face drooping",
        "slurred speech",
        "severe bleeding",
        "seizure",
        "confused",
        "confusion",
        "not urinating",
        "no wet diaper",
        "severe dehydration",
        "blue lips",
        "suicidal",
        "overdose"
    ]

    query_lower = query.lower()

    for keyword in emergency_keywords:
        if keyword in query_lower:
            return True

    return False

In [11]:
def get_safety_instructions(intent):
    if intent == "emergency":
        return """
Safety instructions:
- Treat this as potentially urgent.
- Clearly advise the user to seek emergency medical care immediately.
- Do not give a definite diagnosis.
- Keep the response brief and safety-focused.
"""

    elif intent == "symptom_question":
        return """
Safety instructions:
- Do not diagnose the user.
- Explain possible related information only from sources.
- Encourage consulting a healthcare professional.
- Mention urgent care if symptoms sound severe.
"""

    elif intent == "medication_question":
        return """
Safety instructions:
- Do not prescribe medication or dosage.
- Do not tell the user to start, stop, or change medicine.
- Advise consulting a healthcare professional or pharmacist.
- Use only retrieved source information.
"""

    elif intent == "treatment_question":
        return """
Safety instructions:
- Explain general treatment or management from sources.
- Do not give personal medical instructions.
- Encourage professional medical advice.
"""

    else:
        return """
Safety instructions:
- Use only retrieved sources.
- Do not diagnose.
- Recommend consulting a healthcare professional when appropriate.
"""

In [12]:
def handle_greeting(query):
    return "Hello! I am a medical document assistant. You can ask me health-related questions, and I will answer using the available medical documents."

def handle_out_of_scope(query):
    return "This question is outside the scope of the medical documents. Please ask a health-related question."

In [13]:
def expand_query(query):
    prompt = f"""
You are a medical query expansion assistant for a RAG retriever.

Your job is to convert the user's question into a better search query.

Rules:
- Do not answer the question.
- Keep the original meaning.
- Keep common patient words and add medical synonyms.
- Prefer short keyword-style phrases, not a full sentence.
- Use both patient-friendly words and medical terms.
- Add only highly relevant medical terms.
- Do not add too many possible diseases.
- If the question is not medical, return the original question unchanged.
- Return only one line.

User Question:
{query}

Expanded Search Query:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [14]:
def retrieve_docs(query, top_k=3, fetch_k=10, use_query_expansion=True):
    if use_query_expansion:
        search_query = expand_query(query)
    else:
        search_query = query

    query_embedding = embedding_model.encode([search_query], convert_to_numpy=True)
    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, fetch_k)

    results = []
    seen_questions = set()

    for score, idx in zip(scores[0], indices[0]):
        doc = small_docs[idx]

        question_key = doc["question"].strip().lower()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "score": float(score),
            "expanded_query": search_query,
            "searched_query": search_query
        })

        if len(results) == top_k:
            break

    return results

In [15]:
def assess_retrieval_quality(retrieved_docs, threshold=0.40, min_good_sources=1):
    if len(retrieved_docs) == 0:
        return {
            "is_good": False,
            "reason": "No documents were retrieved.",
            "good_sources_count": 0,
            "best_score": 0.0
        }

    good_sources = [
        doc for doc in retrieved_docs
        if doc["score"] >= threshold
    ]

    best_score = retrieved_docs[0]["score"]

    if len(good_sources) >= min_good_sources:
        return {
            "is_good": True,
            "reason": "Retrieval quality is good enough.",
            "good_sources_count": len(good_sources),
            "best_score": best_score
        }

    return {
        "is_good": False,
        "reason": f"Only {len(good_sources)} sources passed the threshold.",
        "good_sources_count": len(good_sources),
        "best_score": best_score
    }


def filter_relevant_sources(retrieved_docs, min_score=0.50):
    return [
        doc for doc in retrieved_docs
        if doc["score"] >= min_score
    ]

In [16]:
def rewrite_query_for_correction(query, previous_expanded_query=None):
    prompt = f"""
You are a medical query rewriting assistant for a corrective RAG system.

The previous retrieval was weak. Rewrite the user's question into a better medical search query.

Rules:
- Do not answer the question.
- Keep the original meaning.
- Keep common patient words and add medical synonyms.
- Use simple keyword-style phrases.
- Include important symptoms, disease names, body parts, and medical terms.
- Do not add rare diseases unless clearly mentioned by the user.
- If the question is not medical, return the original question unchanged.
- Return only one line.

Original User Question:
{query}

Previous Expanded Query:
{previous_expanded_query}

Corrected Search Query:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


def search_vector_db(search_query, top_k=3, fetch_k=10):
    query_embedding = embedding_model.encode([search_query], convert_to_numpy=True)
    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, fetch_k)

    results = []
    seen_questions = set()

    for score, idx in zip(scores[0], indices[0]):
        doc = small_docs[idx]

        question_key = doc["question"].strip().lower()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "score": float(score),
            "searched_query": search_query
        })

        if len(results) == top_k:
            break

    return results


def corrective_retrieve(
    query,
    top_k=3,
    fetch_k=10,
    threshold=0.40,
    max_corrections=1,
    use_query_expansion=True
):
    correction_count = 0

    retrieved_docs = retrieve_docs(
        query=query,
        top_k=top_k,
        fetch_k=fetch_k,
        use_query_expansion=use_query_expansion
    )

    expanded_query = retrieved_docs[0].get("expanded_query", query) if len(retrieved_docs) > 0 else query

    quality = assess_retrieval_quality(
        retrieved_docs=retrieved_docs,
        threshold=threshold,
        min_good_sources=1
    )

    while not quality["is_good"] and correction_count < max_corrections:
        correction_count += 1

        corrected_query = rewrite_query_for_correction(
            query=query,
            previous_expanded_query=expanded_query
        )

        corrected_docs = search_vector_db(
            search_query=corrected_query,
            top_k=top_k,
            fetch_k=fetch_k
        )

        for doc in corrected_docs:
            doc["expanded_query"] = expanded_query
            doc["corrected_query"] = corrected_query

        retrieved_docs = corrected_docs

        quality = assess_retrieval_quality(
            retrieved_docs=retrieved_docs,
            threshold=threshold,
            min_good_sources=1
        )

    return {
        "retrieved_docs": retrieved_docs,
        "quality": quality,
        "correction_count": correction_count
    }

In [17]:
def generate_answer(query, retrieved_docs, intent):
    context = "\n\n".join([
        f"[Source {i+1}]\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    safety_instructions = get_safety_instructions(intent)

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the provided sources.

General rules:
1. Use only the information from the provided sources.
2. Add citations in the answer using [Source 1], [Source 2], etc.
3. If the sources do not contain enough information, say:
   "The provided medical documents do not contain enough information."
4. Do not diagnose the user.
5. Do not prescribe medicine or dosage.

{safety_instructions}

Sources:
{context}

User question:
{query}

Answer with citations:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [18]:
def rag_chatbot(
    query,
    top_k=3,
    threshold=0.40,
    min_source_score=0.50,
    use_query_expansion=True,
    max_corrections=1
):
    print("Question:")
    print(query)

    intent = classify_intent(query)

    if detect_emergency(query):
        intent = "emergency"

    print("\nIntent:")
    print(intent)

    if intent == "greeting":
        print("\nAnswer:")
        print(handle_greeting(query))
        return

    if intent == "out_of_scope":
        print("\nAnswer:")
        print(handle_out_of_scope(query))
        return

    retrieval_output = corrective_retrieve(
        query=query,
        top_k=top_k,
        fetch_k=10,
        threshold=threshold,
        max_corrections=max_corrections,
        use_query_expansion=use_query_expansion
    )

    retrieved_docs = retrieval_output["retrieved_docs"]
    quality = retrieval_output["quality"]
    correction_count = retrieval_output["correction_count"]

    if len(retrieved_docs) > 0:
        print("\nExpanded Query:")
        print(retrieved_docs[0].get("expanded_query", query))

        if correction_count > 0:
            print("\nCorrected Query:")
            print(retrieved_docs[0].get("corrected_query", "No corrected query available."))

    print("\nRetrieval Quality:")
    print("Best Score:", quality["best_score"])
    print("Good Sources Count:", quality["good_sources_count"])
    print("Correction Count:", correction_count)
    print("Quality Reason:", quality["reason"])

    if not quality["is_good"]:
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")
        return

    filtered_docs = filter_relevant_sources(
        retrieved_docs,
        min_score=min_source_score
    )

    if len(filtered_docs) == 0:
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")
        print("\nReason:")
        print("Retrieved documents were below the minimum source score.")
        return

    answer = generate_answer(query, filtered_docs, intent)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for i, doc in enumerate(filtered_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Similarity Score:", doc["score"])
        print("Searched Query:", doc.get("searched_query", "N/A"))

In [19]:
test_questions = [
    "Hi",
    "What are the symptoms of diabetes?",
    "How is high blood pressure treated?",
    "I feel strange after taking diabetes medicine but I did not eat.",
    "I have chest tightness, sweating, and pain going to my left arm. What should I do?",
    "Baby has diarrhea and dry mouth and no wet diaper.",
    "Who won the FIFA World Cup?",
    "How do I train a machine learning model?"
]

for q in test_questions:
    print("="*100)
    rag_chatbot(q)

Question:
Hi

Intent:
greeting

Answer:
Hello! I am a medical document assistant. You can ask me health-related questions, and I will answer using the available medical documents.
Question:
What are the symptoms of diabetes?

Intent:
symptom_question

Expanded Query:
diabetes symptoms, diabetic signs, hyperglycemia effects, high blood sugar symptoms

Retrieval Quality:
Best Score: 0.7068459391593933
Good Sources Count: 3
Correction Count: 0
Quality Reason: Retrieval quality is good enough.

Answer:
The symptoms of diabetes include being very thirsty, urinating often, feeling very hungry, feeling very tired, losing weight without trying, sores that heal slowly, dry, itchy skin, feelings of pins and needles in your feet, losing feeling in your feet, and blurry eyesight [Source 1]. Additionally, some people with diabetes may experience extreme thirst or hunger, a frequent need to urinate and/or fatigue, and some may lose weight without trying [Source 2]. It's also worth noting that some p